# Eugene's Integral Computations 

## Import Packages

In [3]:
using HCubature # provides adaptive numerical integration 

using Test # provides testing functionality 
using Plots 

## Section 1: Introduction 

We describe here how we wish to proceed. 

### Bottom-up Approach 

1. implement the testing by comparing symbolic and numerical evaluation of the integrals; 

### Top-down Approach 

1. assume two prism to be given;
2. reduce 6D integrals to sum of 4D integrals (how many exactly?) using Euler's theorem for homogeneous functions;
3. evaluate the 4D integrals numerically by some adaptive quadrature;
     

## Section 2: Integrals 

In [45]:
function f1(h,eta)
    return 1/eta^2*(sqrt(h^2+eta^2)-h)
end 

function f2(h,eta)
    return sqrt(h^2+eta^2)/(2*eta^2)-h^2/(2*eta^3)*asinh(eta/h)
end 

function f3(h,eta)
    return (2*h^3+(h^2+eta^2)^(3/2)-3*h^2*sqrt(h^2+eta^2))/(3*eta^4)
end 

function I8integrand(s,h,R0)
    arg = sqrt(R0^2+s^2)
    return f1(h,arg) 
end

function I8(s,h,R0)
    t0 = sqrt(h^2+R0^2)
    return I14(s,h,R0) + asinh(s/t0) - h/R0*atan(s/R0)
end

function I9(s,h,R0)
    t0 = sqrt(h^2+R0^2)
    return sqrt(t0^2+s^2)/h-asinh(h/sqrt(R0^2+s^2))-.5*log(R0^2+s^2) 
end

function I9integrand(s,h,R0)
    arg = sqrt(R0^2+s^2)
    return s/h*f1(h,arg) 
end

function I10(s,h,R0)
    t0 = sqrt(h^2+R0^2)
    return t0^2/(2*R0^2)*asinh(s/t0)-(h^2*s)/(2*R0^2*sqrt(R0^2+s^2))*asinh(sqrt(R0^2+s^2)/h)
end

function I10integrand(s,h,R0)
    arg = sqrt(R0^2+s^2)
    return f2(h,arg) 
end

function I11(s,h,R0)
    t0 = sqrt(h^2+R0^2)
    # return sqrt(t0^2+s^2)/(2*h)-asinh(h/sqrt(R0^2+s^2))+h*asinh(sqrt(R0^2+s^2)/h)/(2*sqrt(R0^2+s^2))+.5*asinh(h/sqrt(R0^2+s^2))
    return sqrt(t0^2+s^2)/(2*h)+.5*log((sqrt(t0^2+s^2)-h)/(sqrt(t0^2+s^2)+h))+h*asinh(sqrt(R0^2+s^2)/h)/(2*sqrt(R0^2+s^2))+.5*asinh(h/sqrt(R0^2+s^2))
end

function I11integrand(s,h,R0)
    arg = sqrt(R0^2+s^2)
    return s/h*f2(h,arg) 
end

function I12(s,h,R0) 
    return I14(s,h,R0)/3+I15(s,h,R0)/3+I18(s,h,R0)+2*I19(s,h,R0)/3 
end

function I12integrand(s,h,R0)
    arg = sqrt(R0^2+s^2)
    return f3(h,arg) 
end

function I13(s,h,R0)
    t0 = sqrt(h^2+R0^2)
    value = ((t0^2+s^2)^(3/2)/h-h^2)/(3*(R0^2+s^2))
    return value-asinh(h/sqrt(R0^2+s^2))/3 
end

function I13integrand(s,h,R0)
    arg = sqrt(R0^2+s^2)
    return s/h*f3(h,arg) 
end

function I14(s,h,R0)
    t0 = sqrt(h^2+R0^2)
    argplus = sqrt(t0^2+s^2)+h
    argmin  = sqrt(t0^2+s^2)-h
    return sign(s)*(I17plus(argmin,h,R0) - I17min(argplus,h,R0)) 
end

function I15(s,h,R0)
    t0 = sqrt(h^2+R0^2)
    return asinh(s/t0)
end

function I16(s,h,R0)
    t0 = sqrt(h^2+R0^2)
    return h/R0*atan(s/R0)
end

function I17plus(x,h,R0)
    t0 = sqrt(h^2+R0^2)
    arg = -R0^2/(x*t0)+h/t0
    return h/(2*R0)*asin(arg) 
end 

function I17min(x,h,R0)
    t0 = sqrt(h^2+R0^2)
    arg  = -R0^2/(x*t0)-h/t0
    return h/(2*R0)*asin(arg)  
end

function I18(s,h,R0)
    t0 = sqrt(h^2+R0^2)
    return h^3/(3*R0^3)*(R0*s/(R0^2+s^2)+atan(s/R0)) 
end

function I19(s,h,R0)
    t0 = sqrt(h^2+R0^2)
    value = -(h^2*s*sqrt(t0^2+s^2))/(2*R0^2*(R0^2+s^2))
    value += h/(2*R0)*(1-h^2/R0^2)*atan(h*s/(R0*sqrt(t0^2+s^2)))
    return value-I14(s,h,R0)
end;


## Section 3: Testing Function Evaluations

In [46]:
f3(.5,.5)

0.39052429175126946

## Section 4: Testing the Integrals (I11 and I13 currently fail)

In [47]:
h = .5
R0 = .5
s1 = .1; s2 = 1. 
I11(s2,h,R0) - I11(s1,h,R0) 

0.6340324251780942

In [48]:
hcubature(s -> I11integrand(s[1],h,R0) , (s1,), (s2,))

(0.41699852046271785, 1.7793388984443936e-10)